# Dataset inspection: participant-level opinion-dynamics episodes

This notebook loads all six unified datasets through `OpinionDynamicsDataLoader`, builds participant-perspective episodes, and inspects stance proxies, visibility, SELF/OTHER assignment, and chat formatting.

**Scope:** data infrastructure and inspection only — no training.

## 1. Load all six datasets

In [1]:
from pathlib import Path
from opinion_dynamics import OpinionDynamicsDataLoader

REPO = Path('..').resolve()
loader = OpinionDynamicsDataLoader.from_config(REPO / 'configs' / 'datasets.yaml', repo_root=REPO)
episodes = loader.load()
print(f'Loaded {len(episodes)} participant episodes from {loader.summary().original_record_count} records')
print(loader.summary())

Loaded 12083 participant episodes from 15010 records
=== Opinion Dynamics Corpus Summary ===
Original records          : 15010
Participant episodes      : 12083
Unique participants       : 12083
Records / dataset         : {'ConspiracyAI': 1749, 'DEBATE': 263, 'PersuasionTrace': 353, 'AnthropicPersuasion': 3939, 'CMV': 8106, 'DebateGPT': 600}
Episodes / dataset        : {'ConspiracyAI': 1749, 'DEBATE': 1052, 'PersuasionTrace': 353, 'AnthropicPersuasion': 3939, 'CMV': 4240, 'DebateGPT': 750}
Initial ratings           : 12083
Final ratings             : 12083
Both ratings              : 12083
Binary proxy              : 4240
Continuous                : 7843
With initial context      : 7041
Without initial context   : 5042
Control episodes          : 125
Splits                    : {'train': 9177, 'val': 1246, 'test': 1660}
Structure types           : {'human_llm_dialogue': 2102, 'multi_party_rounds': 1052, 'one_shot_argument': 3939, 'reddit_thread': 4240, 'debate_phases': 750}
Conversati

## 2. Comprehensive dataset summary

In [2]:
summary = loader.summary()
print(summary)
print('\nExclusions (first 10):')
for ex in summary.exclusions[:10]:
    print(' ', ex)

=== Opinion Dynamics Corpus Summary ===
Original records          : 15010
Participant episodes      : 12083
Unique participants       : 12083
Records / dataset         : {'ConspiracyAI': 1749, 'DEBATE': 263, 'PersuasionTrace': 353, 'AnthropicPersuasion': 3939, 'CMV': 8106, 'DebateGPT': 600}
Episodes / dataset        : {'ConspiracyAI': 1749, 'DEBATE': 1052, 'PersuasionTrace': 353, 'AnthropicPersuasion': 3939, 'CMV': 4240, 'DebateGPT': 750}
Initial ratings           : 12083
Final ratings             : 12083
Both ratings              : 12083
Binary proxy              : 4240
Continuous                : 7843
With initial context      : 7041
Without initial context   : 5042
Control episodes          : 125
Splits                    : {'train': 9177, 'val': 1246, 'test': 1660}
Structure types           : {'human_llm_dialogue': 2102, 'multi_party_rounds': 1052, 'one_shot_argument': 3939, 'reddit_thread': 4240, 'debate_phases': 750}
Conversation length       : {'min': 1, 'max': 47, 'mean': 5.185

## 3. Inspect at least three examples from each dataset

In [3]:
from collections import defaultdict

by_ds = defaultdict(list)
for ep in episodes:
    by_ds[ep.source.dataset].append(ep)

for ds, eps in sorted(by_ds.items()):
    print('\n' + '#' * 72)
    print(f'# {ds} — showing 3 / {len(eps)} episodes')
    print('#' * 72)
    for ep in eps[:3]:
        print(loader.inspect(ep, max_text=180))
        print()


########################################################################
# AnthropicPersuasion — showing 3 / 3939 episodes
########################################################################
EPISODE  : anthropicpersuasion:r0000:T
SOURCE   : AnthropicPersuasion / r0000
SPLIT    : test
LICENSE  : CC-BY-NC-SA-4.0

AGENT    : T (kind=human, role=target)

PROPOSITION:
  text       : Governments and technology companies must do more to protect online privacy and security.
  provenance : claim

INITIAL:
  stance=100.0 raw=7.0 scale=[1, 7] type=continuous prov=self_reported
  context: (none)

DISCUSSION (1 visible turns):
  [0] round=1 channel=dialogue AI (other/llm) -> ['T'] reply_to=None
      It's time for governments and tech companies to act. Our privacy and security online are under attack. Every day, our personal data is collected, shared, and exploited without o...

FINAL:
  stance=100.0 raw=7.0 scale=[1, 7] type=continuous prov=self_reported
  context: (none)

OUTCOME: changed=F

## 4. Initial vs final context availability

In [4]:
import pandas as pd

rows = []
for ds, eps in sorted(by_ds.items()):
    rows.append({
        'dataset': ds,
        'episodes': len(eps),
        'has_initial_context': sum(e.metadata.has_initial_context for e in eps),
        'has_final_context': sum(e.metadata.has_final_context for e in eps),
        'has_discussion': sum(1 for e in eps if e.discussion.visible_turns),
        'has_initial_rating': sum(e.metadata.has_initial_rating for e in eps),
        'has_final_rating': sum(e.metadata.has_final_rating for e in eps),
        'rating_only_or_missing_init': sum(
            1 for e in eps
            if e.initial.context and e.initial.context.type in {'rating_only', 'missing'}
        ),
    })
pd.DataFrame(rows)

,dataset,episodes,has_initial_context,has_final_context,has_discussion,has_initial_rating,has_final_rating,rating_only_or_missing_init
0,AnthropicPersuasion,3939,0,0,3939,3939,3939,0
1,CMV,4240,4240,58,4240,4240,4240,0
2,ConspiracyAI,1749,1749,0,1749,1749,1749,0
3,DEBATE,1052,1052,1052,1052,1052,1052,0
4,DebateGPT,750,0,0,750,750,750,0
5,PersuasionTrace,353,0,0,353,353,353,0


## 5. CMV binary stance proxies

In [5]:
cmv = loader.filter(dataset='CMV')
success = [e for e in cmv if e.outcome.changed is True][:3]
fail = [e for e in cmv if e.outcome.changed is False][:3]
print('Successful Δ examples (expect initial=100, final=0):')
for e in success:
    print(f'  {e.episode_id}: init={e.initial.stance} final={e.final.stance} type={e.initial.stance_type} prov={e.initial.stance_provenance}')
    print(f'    final_context: {(e.final.context.text or "")[:120]!r}')
print('\nUnsuccessful examples (expect initial=100, final=100):')
for e in fail:
    print(f'  {e.episode_id}: init={e.initial.stance} final={e.final.stance} changed={e.outcome.changed}')
print('\nDetailed inspection of one successful CMV episode:')
print(loader.inspect(success[0], max_text=220))

Successful Δ examples (expect initial=1, final=0):
  cmv:t3_2ro9ux/t1_cnhplrm:OP: init=1.0 final=0.0 type=binary_proxy prov=dataset_outcome_proxy
    final_context: ''
  cmv:t3_2ro0ti/t1_cnhuxye:OP: init=1.0 final=0.0 type=binary_proxy prov=dataset_outcome_proxy
    final_context: ''
  cmv:t3_2rnr30/t1_cni3o83:OP: init=1.0 final=0.0 type=binary_proxy prov=dataset_outcome_proxy
    final_context: ''

Unsuccessful examples (expect initial=1, final=1):

Detailed inspection of one successful CMV episode:
EPISODE  : cmv:t3_2ro9ux/t1_cnhplrm:OP
SOURCE   : CMV / t3_2ro9ux/t1_cnhplrm
SPLIT    : train
LICENSE  : Reddit user content; research use (Tan et al. 2016 / ConvoKit)

AGENT    : OP (kind=human, role=op)

PROPOSITION:
  text       : Anything that is man-made is natural.
  provenance : op_title

INITIAL:
  stance=1.0 raw=None scale=None type=binary_proxy prov=dataset_outcome_proxy
  context type=participant_authored speaker=OP/self source=human_authored
  context text: I can't remember the

## 6. PersuasionTrace first/last stance extraction

In [6]:
pt = loader.filter(dataset='PersuasionTrace')
main = [e for e in pt if not e.metadata.control][:2]
ctrl = [e for e in pt if e.metadata.control][:2]
for label, group in [('main', main), ('control', ctrl)]:
    print(f'\n=== {label} ===')
    for e in group:
        trace = e.extra.get('opinion_trace') or []
        print(f'{e.episode_id}')
        print(f'  initial={e.initial.stance:.4f} final={e.final.stance:.4f} change={e.outcome.change_100}')
        print(f'  trace_len={len(trace)} (intermediates kept only in extra)')
        print(f'  control={e.metadata.control}')
        if e.metadata.limitations:
            print(f'  limitations: {e.metadata.limitations[0][:120]}')


=== main ===
persuasiontrace:a5f211fc979cddec:T
  initial=62.2526 final=21.5026 change=-40.75
  trace_len=3 (intermediates kept only in extra)
  control=False
persuasiontrace:ad2c4bc45e8b91b4:T
  initial=75.5000 final=64.2539 change=-11.2461
  trace_len=6 (intermediates kept only in extra)
  control=False

=== control ===
persuasiontrace:6288e8e8cd76a675:T
  initial=66.2500 final=71.5000 change=5.25
  trace_len=3 (intermediates kept only in extra)
  control=True
  limitations: CONTROL dialogue: proposition.text is the discussion topic ('Dogs are better than cats.'). Belief ratings were collected
persuasiontrace:25e0dbb4370dd7f6:T
  initial=70.0000 final=79.2500 change=9.25
  trace_len=4 (intermediates kept only in extra)
  control=True
  limitations: CONTROL dialogue: proposition.text is the discussion topic ('Dogs are better than cats.'). Belief ratings were collected


## 7. All four DEBATE participant perspectives

In [7]:
debate_eps = loader.filter(dataset='DEBATE', split='train')
# Pick a record with intact speakers
from collections import Counter
by_orig = defaultdict(list)
for e in debate_eps:
    by_orig[e.source.original_id].append(e)
record_id = next(oid for oid, eps in by_orig.items() if len(eps) == 4)
siblings = by_orig[record_id]
print(loader.inspect_perspective(siblings[0].episode_id, max_text=120))

=== Perspective comparison for DEBATE:01JTGYEQWZY3R160PPY0GEBZSW ===
4 participant episode(s)

--- Agent P1 (debate:01JTGYEQWZY3R160PPY0GEBZSW:P1) ---
  initial=40.0 final=40.0 visible_turns=31 excluded=13
  speaker_roles: {'self': 13, 'other': 18}
  id->role: {'P1': 'self', 'P4': 'other', 'P3': 'other', 'P2': 'other'}
    [0] P1/self ch=public: I don't think citizens should go on major issues that's what we elect our senators and congressmen They are our Voice...
    [1] P4/other ch=public: I don’t think direct citizen voting on all major national issues is a good way to govern the U.S. Most policies are t...
    [2] P3/other ch=public: Ensure citizen voices count, not the will of one person who turns their backs on their constituents for money. Everyo...
    [3] P2/other ch=public: it is a right way of governance and also show that the citizens are listen too and their voices are con
    [4] P1/self ch=private: If you get the right people elected they will represent your voice
    ..

## 8. SELF/OTHER speaker assignments

In [8]:
for e in siblings:
    roles = Counter((t.speaker_id, t.speaker_role) for t in e.discussion.visible_turns)
    print(f'Agent {e.agent.participant_id}:')
    for (sid, role), n in sorted(roles.items()):
        print(f'  {sid:>3} -> {role:7} x{n}')
    print()

Agent P1:
   P1 -> self    x13
   P2 -> other   x7
   P3 -> other   x5
   P4 -> other   x6

Agent P2:
   P1 -> other   x6
   P2 -> self    x11
   P3 -> other   x5
   P4 -> other   x5

Agent P3:
   P1 -> other   x6
   P2 -> other   x5
   P3 -> self    x10
   P4 -> other   x5

Agent P4:
   P1 -> other   x7
   P2 -> other   x5
   P3 -> other   x6
   P4 -> self    x10



## 9. Original conversations vs participant-specific histories

In [9]:
print(loader.compare_original_and_episode(siblings[0].episode_id))
print()
for e in siblings:
    print(loader.inspect_visibility(e.episode_id))
    print()

=== Original vs Episode: debate:01JTGYEQWZY3R160PPY0GEBZSW:P1 ===

SOURCE RECORD:
  id=debate:01JTGYEQWZY3R160PPY0GEBZSW split=train
  dataset=DEBATE
  n_parties=4 n_turns=44
  structure=multi_party_rounds control=False
  proposition: A democratic system where citizens, not elected officials, vote directly on major national issues to decide what becomes law is a good way of governing the US

TRACKED PARTY (original):
  id=P1 kind=human role=participant tracked=True
  initial raw=3 rating_100=40.0 stmt_prov=human_private
  initial statement: To the officials is the best way to go not having citizens vote on major issues it would just be too chaotic
  final raw=3 rating_100=40.0 stmt_prov=human_private
  final statement: While the sounds ideal to have citizens vote on major issues to become law I don't think it's practical
  opinion.trace length=1

GENERATED EPISODE:
  agent=P1
  initial stance=40.0 (raw=3.0, type=continuous)
  final stance=40.0 (raw=3.0, type=continuous)
  visible_turns

## 10. AnthropicPersuasion one-shot arguments

In [10]:
ai_eps = [e for e in loader.filter(dataset='AnthropicPersuasion')
          if e.discussion.visible_turns and e.discussion.visible_turns[0].speaker_id == 'AI'][:2]
human_eps = [e for e in loader.filter(dataset='AnthropicPersuasion')
             if e.discussion.visible_turns and e.discussion.visible_turns[0].speaker_id == 'P1'][:2]
print('AI-authored arguments:')
for e in ai_eps:
    t = e.discussion.visible_turns[0]
    print(f'  {e.episode_id}: speaker={t.speaker_id}/{t.speaker_kind}/{t.speaker_role} init={e.initial.stance} final={e.final.stance}')
    print(f'    arg[:160]={t.text[:160]!r}')
print('\nHuman-authored arguments:')
for e in human_eps:
    t = e.discussion.visible_turns[0]
    print(f'  {e.episode_id}: speaker={t.speaker_id}/{t.speaker_kind}/{t.speaker_role}')
print('\nFull inspection:')
print(loader.inspect(ai_eps[0], max_text=250))

AI-authored arguments:
  anthropicpersuasion:r0000:T: speaker=AI/llm/other init=100.0 final=100.0
    arg[:160]="It's time for governments and tech companies to act. Our privacy and security online are under attack. Every day, our personal data is collected, shared, and ex"
  anthropicpersuasion:r0001:T: speaker=AI/llm/other init=100.0 final=100.0
    arg[:160]="In today's hyper-connected world, our personal information and digital lives are more vulnerable than ever before. Governments and technology companies have a m"

Human-authored arguments:
  anthropicpersuasion:r3360:T: speaker=P1/human/other
  anthropicpersuasion:r3361:T: speaker=P1/human/other

Full inspection:
EPISODE  : anthropicpersuasion:r0000:T
SOURCE   : AnthropicPersuasion / r0000
SPLIT    : test
LICENSE  : CC-BY-NC-SA-4.0

AGENT    : T (kind=human, role=target)

PROPOSITION:
  text       : Governments and technology companies must do more to protect online privacy and security.
  provenance : claim

INITIAL:
  stance=

## 11. DebateGPT configurations

In [11]:
dg = loader.filter(dataset='DebateGPT')
from collections import defaultdict
by_treat = defaultdict(list)
for e in dg:
    by_treat[e.extra.get('treatment')].append(e)
for treat, eps in sorted(by_treat.items()):
    agents = Counter(e.agent.participant_id for e in eps)
    print(f'{treat}: {len(eps)} episodes, agents={dict(agents)}')
    e = eps[0]
    print(f'  sample {e.episode_id}: turns={len(e.discussion.visible_turns)} '
          f'phases={[t.annotations.get("phase") for t in e.discussion.visible_turns]}')
    print(f'  assumption: {e.metadata.assumptions[-1] if e.metadata.assumptions else None}')
    print()

Human-AI: 150 episodes, agents={'P1': 150}
  sample debategpt:3:P1: turns=6 phases=['argument', 'argument', 'rebuttal', 'rebuttal', 'conclusion', 'conclusion']
  assumption: Within each phase, the Pro turn is listed before the Con turn by convention; this is not verified chronological ordering (phases were written in parallel).

Human-AI, personalized: 150 episodes, agents={'P1': 150}
  sample debategpt:1:P1: turns=6 phases=['argument', 'argument', 'rebuttal', 'rebuttal', 'conclusion', 'conclusion']
  assumption: Within each phase, the Pro turn is listed before the Con turn by convention; this is not verified chronological ordering (phases were written in parallel).

Human-Human: 300 episodes, agents={'P1': 150, 'P2': 150}
  sample debategpt:4:P1: turns=6 phases=['argument', 'argument', 'rebuttal', 'rebuttal', 'conclusion', 'conclusion']
  assumption: Within each phase, the Pro turn is listed before the Con turn by convention; this is not verified chronological ordering (phases were wr

## 12. Stance-prediction chat messages

In [12]:
ep = siblings[0]
messages = ep.to_stance_prediction_messages()
for m in messages:
    print('=' * 40, m['role'].upper(), '=' * 40)
    print(m['content'][:2000])
    if len(m['content']) > 2000:
        print('... [truncated]')
print('\nTarget final stance (not in messages):', ep.final.stance)

======================================== SYSTEM ========================================
You are simulating participant P1.

SELF refers to P1.
OTHER refers to all other participants.

Preserve the distinction between participants.
Only use information available to P1.
======================================== USER ========================================
PROPOSITION:
A democratic system where citizens, not elected officials, vote directly on major national issues to decide what becomes law is a good way of governing the US

INITIAL STANCE:
40.0 (type=continuous, provenance=self_reported)

INITIAL CONTEXT:
[SELF: P1]
To the officials is the best way to go not having citizens vote on major issues it would just be too chaotic

DISCUSSION:

[SELF: P1]
(channel=public, round=1)
I don't think citizens should go on major issues that's what we elect our senators and congressmen They are our Voice so elect them accordingly

[OTHER: P4]
(channel=public, round=1)
I don’t think direct citizen voti

## 13. Next-response-generation chat messages

In [13]:
from opinion_dynamics.formatting import ResponseGenerationFormatter
fmt = ResponseGenerationFormatter()
candidates = fmt.candidate_target_indices(ep)
print('Candidate SELF turn indices:', candidates)
target_idx = candidates[min(2, len(candidates)-1)]
messages, target = ep.to_response_generation_messages(target_turn_idx=target_idx)
for m in messages:
    print('=' * 40, m['role'].upper(), '=' * 40)
    print(m['content'][:1800])
print('\n=== EXPECTED ASSISTANT TARGET (separate) ===')
print(target)
assert target not in messages[1]['content']
print('\nVerified: target response is not leaked into the prompt.')

Candidate SELF turn indices: [0, 4, 6, 8, 14, 18, 20, 22, 31, 33, 35, 37, 39]
======================================== SYSTEM ========================================
You are simulating participant P1.

SELF refers to P1.
OTHER refers to all other participants.

Preserve the distinction between participants.
Only use information available to P1.
======================================== USER ========================================
PROPOSITION:
A democratic system where citizens, not elected officials, vote directly on major national issues to decide what becomes law is a good way of governing the US

INITIAL CONTEXT:
[SELF: P1]
To the officials is the best way to go not having citizens vote on major issues it would just be too chaotic

DISCUSSION:

[SELF: P1]
(channel=public, round=1)
I don't think citizens should go on major issues that's what we elect our senators and congressmen They are our Voice so elect them accordingly

[OTHER: P4]
(channel=public, round=1)
I don’t think direct 

## 14. Missing ratings and contexts

In [14]:
missing = {
    'no_initial_rating': loader.filter(has_initial_rating=False),
    'no_final_rating': loader.filter(has_final_rating=False),
    'no_initial_context': loader.filter(has_initial_context=False),
}
for k, v in missing.items():
    by = Counter(e.source.dataset for e in v)
    print(f'{k}: {len(v)} -> {dict(by)}')

no_initial_rating: 0 -> {}
no_final_rating: 0 -> {}
no_initial_context: 5042 -> {'PersuasionTrace': 353, 'AnthropicPersuasion': 3939, 'DebateGPT': 750}


## 15–16. Export, reload, and consistency check

In [15]:
out = REPO / 'data' / 'processed' / 'episodes_sample.jsonl'
# Export a manageable sample for the notebook demo
sample = []
for ds in sorted(by_ds):
    sample.extend(by_ds[ds][:5])
loader.export_jsonl(out, sample)
reloaded = OpinionDynamicsDataLoader.load_episodes_jsonl(out)
print(f'Exported {len(sample)} episodes -> {out}')
print(f'Reloaded {len(reloaded)} episodes')
mismatches = 0
for a, b in zip(sample, reloaded):
    if a.to_dict() != b.to_dict():
        mismatches += 1
        print('Mismatch:', a.episode_id)
print(f'Consistency mismatches: {mismatches}')
assert mismatches == 0
print('Reload is bit-identical to exported episode dicts.')

Exported 30 episodes -> /Users/z5517269/Desktop/Repos/opinion-dynamics/data/processed/episodes_sample.jsonl
Reloaded 30 episodes
Consistency mismatches: 0
Reload is bit-identical to exported episode dicts.


## Optional: Hugging Face Dataset conversion

In [16]:
hf = loader.to_huggingface(sample)
print(hf)
print(hf[0]['episode_id'], hf[0]['agent'])

Dataset({
    features: ['episode_id', 'source', 'agent', 'proposition', 'initial', 'discussion', 'final', 'outcome', 'metadata', 'extra'],
    num_rows: 30
})
anthropicpersuasion:r0000:T {'participant_id': 'T', 'kind': 'human', 'role': 'target'}
